# Correlation contribution to the kinetic energy from virial theorem
**Author:** Egor Trushin  
**Date created:** 15/04/2025  
**Last modified:** 11/08/2025

## References
- F.G. Cruz, K.-C. Lam, K. Burke Exchange-Correlation Energy Density from Virial Theorem. J. Phys. Chem. A 1998, 102, 4911-4917 https://doi.org/10.1021/jp980950v
- J.I. Rodríguez, P.W. Ayers, A.W. Götz, F.L. Castillo-Alvarado Virial theorem in the Kohn–Sham density-functional theory formalism: Accurate calculation of the atomic quantum theory of atoms in molecules energies. J. Chem. Phys. 131, 021101 (2009) https://doi.org/10.1063/1.3160670
- H. Ou-Yang, M. Levy Theorem for exact local exchange potential. Phys. Rev. Lett. 65, 1036 (1990) https://doi.org/10.1103/PhysRevLett.65.1036

## Equations
$$E_c + T_c = - \int d\mathbf{r} \rho(\mathbf{r}) \mathbf{r} \cdot \nabla v_c(\mathbf{r}) = \int d\mathbf{r} (3\rho(\mathbf{r}) + \mathbf{r} \cdot \nabla \rho(\mathbf{r})) v_c(\mathbf{r})$$
$$T_c = - E_c + \int d\mathbf{r} (3\rho(\mathbf{r}) + \mathbf{r} \cdot \nabla \rho(\mathbf{r})) v_c(\mathbf{r})$$

In [1]:
import pyscf
print("PySCF version:", pyscf.__version__)

import numpy as np
from pyscf import gto, dft

PySCF version: 2.10.0


## Functions for evaluation of LDA and GGA potentials

In [2]:
def lda_vxc(mol, coords, dm, xc):
    """Evaluate the LDA exchange-correlation potential on a given real-space grid.

    Args:
      mol: PySCF molecule object
      coords: coordinates of grid on which the potential will be calculated
      dm: density matrix
      xc: xc string, i.e. xc='lda'

    Returns:
      vxc: exchange-correlation potential on grid
    """
    ao_value = dft.numint.eval_ao(mol, coords, deriv=0)
    rho = dft.numint.eval_rho(mol, ao_value, dm, xctype="LDA")
    _, vxc = dft.libxc.eval_xc(xc, rho)[:2]
    return vxc[0]

In [3]:
def gga_vxc(mol, coords, dm, xc):
    """Evaluate the GGA exchange-correlation potential on a given real-space grid.

    Args:
      mol: PySCF molecule object
      coords: coordinates of grid on which the potential will be calculated
      dm: density matrix
      xc: xc string, i.e. xc='pbe'

    Returns:
      vxc: exchange-correlation potential on grid
    """
    ao = dft.numint.eval_ao(mol, coords, deriv=2)
    ao_0, ao_1 = ao[0], ao[1:4]
    ao_2 = np.array(
        [
            [ao[4], ao[5], ao[6]],
            [ao[5], ao[7], ao[8]],
            [ao[6], ao[8], ao[9]],
        ]
    )

    rho_0 = np.einsum("gu, gv, uv -> g", ao_0, ao_0, dm)
    rho_1 = 2 * np.einsum("rgu, gv, uv -> rg", ao_1, ao_0, dm)
    rho_2 = +2 * np.einsum("rwgu, gv, uv -> rwg", ao_2, ao_0, dm) + 2 * np.einsum("rgu, wgv, uv -> rwg", ao_1, ao_1, dm)

    gamma_0 = np.einsum("rg, rg -> g", rho_1, rho_1)
    gamma_1 = 2 * np.einsum("rwg, wg -> rg", rho_2, rho_1)

    _, vxc, fxc, _ = dft.libxc.eval_xc(xc, np.concatenate([rho_0[None, :], rho_1]), deriv=2)
    fr, fg = vxc[:2]
    frr, frg, fgg = fxc[:3]

    vxc = (
        +fr
        - 2 * np.einsum("g, rg, rg -> g", frg, rho_1, rho_1)
        - 2 * np.einsum("g, rg, rg -> g", fgg, gamma_1, rho_1)
        - 2 * np.einsum("g, gr -> g", fg, rho_2.diagonal())
    )
    return vxc

## Reproducing results from *Cruz et al*

In [4]:
systems = ["He", "Be", "Ne"]

funcs = {
    "LSD": ["LDA_X", "LDA_C_VWN_RPA"],
    "PW91": ["GGA_X_PW91", "GGA_C_PW91"],
    "PBE": ["GGA_X_PBE", "GGA_C_PBE"],
    "BLYP": ["GGA_X_B86", "GGA_C_LYP"],
}

In [5]:
results = {}
for s in systems:
    results[s] = {}
    for func in funcs:
        x = funcs[func][0]
        c = funcs[func][1]

        mol = gto.M(atom=f"{s} 0. 0. 0.", basis="cc-pV5Z", symmetry=False)
        mol.verbose = 0

        mf = dft.RKS(mol, xc=f"{x}, {c}").density_fit(auxbasis="aug-cc-pV5Z-RIFIT").run()
        dm = mf.make_rdm1()

        if func in ["PBE", "BLYP", "PW91"]:
            ao = dft.numint.eval_ao(mol, mf.grids.coords, deriv=2)
            rho = dft.numint.eval_rho(mol, ao, dm, xctype="MGGA")
            vc = gga_vxc(mol, mf.grids.coords, dm, f"{c}")
            Integral = np.einsum(
                "n,n->",
                3 * rho[0] + np.einsum("nd,dn->n", mf.grids.coords, rho[1:4]),
                vc * mf.grids.weights,
            )
            ex, vx = dft.libxc.eval_xc(f"{x}", rho[0:4])[:2]
            ec, vc = dft.libxc.eval_xc(f"{c}", rho[0:4])[:2]
            E_x = np.einsum("i,i,i->", ex, rho[0], mf.grids.weights)
            E_c = np.einsum("i,i,i->", ec, rho[0], mf.grids.weights)
        else:
            ao = dft.numint.eval_ao(mol, mf.grids.coords, deriv=1)
            rho = dft.numint.eval_rho(mol, ao, dm, xctype="GGA")
            vc = lda_vxc(mol, mf.grids.coords, dm, f"{c}")
            Integral = np.einsum(
                "n,n->",
                3 * rho[0] + np.einsum("nd,dn->n", mf.grids.coords, rho[1:4]),
                vc * mf.grids.weights,
            )
            ex, vx = dft.libxc.eval_xc(f"{x}", rho[0])[:2]
            ec, vc = dft.libxc.eval_xc(f"{c}", rho[0])[:2]
            E_x = np.einsum("i,i,i->", ex, rho[0], mf.grids.weights)
            E_c = np.einsum("i,i,i->", ec, rho[0], mf.grids.weights)

        T_c = Integral - E_c
        results[s][func] = {"E_x": E_x, "E_c": E_c, "T_c": T_c, "E_c+T_c": E_c + T_c}

In [6]:
print("             LSD       PW91     PBE     BLYP")
for s in systems:
    print(s)
    print(f"E_x        {results[s]['LSD']['E_x']:7.3f}  {results[s]['PW91']['E_x']:7.3f}  {results[s]['PBE']['E_x']:7.3f}  {results[s]['BLYP']['E_x']:7.3f}")
    print(f"E_c        {results[s]['LSD']['E_c']:7.3f}  {results[s]['PW91']['E_c']:7.3f}  {results[s]['PBE']['E_c']:7.3f}  {results[s]['BLYP']['E_c']:7.3f}")
    print(f"T_c        {results[s]['LSD']['T_c']:7.3f}  {results[s]['PW91']['T_c']:7.3f}  {results[s]['PBE']['T_c']:7.3f}  {results[s]['BLYP']['T_c']:7.3f}")
    print(f"E_c+T_c    {results[s]['LSD']['E_c+T_c']:7.3f}  {results[s]['PW91']['E_c+T_c']:7.3f}  {results[s]['PBE']['E_c+T_c']:7.3f}  {results[s]['BLYP']['E_c+T_c']:7.3f}")

             LSD       PW91     PBE     BLYP
He
E_x         -0.863   -1.010   -1.005   -1.020
E_c         -0.149   -0.045   -0.041   -0.044
T_c          0.100    0.037    0.037    0.033
E_c+T_c     -0.049   -0.008   -0.004   -0.011
Be
E_x         -2.292   -2.644   -2.634   -2.670
E_c         -0.298   -0.094   -0.085   -0.095
T_c          0.202    0.073    0.072    0.061
E_c+T_c     -0.096   -0.021   -0.014   -0.034
Ne
E_x        -10.972  -12.084  -12.029  -12.123
E_c         -0.945   -0.379   -0.347   -0.383
T_c          0.679    0.311    0.304    0.283
E_c+T_c     -0.266   -0.068   -0.043   -0.100
